In [2]:
# =============================================================================
# Freedom Insurance — XGBoost Scoring Model FINAL FIXED

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import brentq
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, r2_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import calibration_curve
import xgboost as xgb
import joblib

# =============================================================================
# 0. Параметры
# =============================================================================
TARGET_LR        = 0.70
GROUP1_THRESHOLD = 1.40  
RANDOM_STATE     = 42
N_FOLDS          = 5
MIN_PREMIUM      = 1.0
OUTPUT_DIR       = Path("output_xgb_final")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

# =============================================================================
# 1. Загрузка данных
# =============================================================================
print("=" * 60)
print("1. Загрузка данных")
train = pd.read_csv("train.csv", low_memory=False)
test  = pd.read_csv("test_final.csv", low_memory=False)
print(f"Train: {train.shape} | Test: {test.shape}")

# =============================================================================
# 1b. HOLDOUT SPLIT 
# =============================================================================
train, train_holdout = train_test_split(train, test_size=0.20, random_state=RANDOM_STATE)
train         = train.reset_index(drop=True)
train_holdout = train_holdout.reset_index(drop=True)
print(f"  Train fit: {len(train):,} | Holdout (20%): {len(train_holdout):,}")

# =============================================================================
# 2. Дедупликация + SCORE агрегация
# =============================================================================
score_cols = [c for c in train.columns if c.startswith("SCORE")]
dup_mask   = train.drop(columns=["unique_id"] + score_cols, errors="ignore").duplicated()
print(f"\n2. Дубликатов: {dup_mask.sum()} -> убраны")
train = train[~dup_mask].reset_index(drop=True)

DATASET_YEAR = int(
    pd.to_datetime(train["operation_date"], errors="coerce")
    .dt.year.dropna().mode()[0]
)
print(f"  Год датасета: {DATASET_YEAR}")

score_cols = [c for c in train.columns if c.startswith("SCORE")]
if score_cols:
    score_cols_num  = train[score_cols].apply(pd.to_numeric, errors="coerce")
    score_zero_rate = (score_cols_num == 0).mean()
    score_cols_drop = score_zero_rate[score_zero_rate > 0.95].index.tolist()
    score_cols      = [c for c in score_cols if c not in score_cols_drop]

score_groups = {}
for col in score_cols:
    prefix = "_".join(col.split("_")[:2])
    score_groups.setdefault(prefix, []).append(col)
for df in [train, test]:
    for prefix, cols in score_groups.items():
        df[f"{prefix}_avg"] = df[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
score_avg_cols = [f"{p}_avg" for p in score_groups]
train.drop(columns=score_cols, inplace=True)
test.drop(columns=[c for c in score_cols if c in test.columns], inplace=True)
print(f"  SCORE сжаты: {sum(len(v) for v in score_groups.values())} -> {len(score_avg_cols)} средних")

# =============================================================================
# 3. Целевая переменная
# =============================================================================
train["claim_amount"] = pd.to_numeric(train["claim_amount"], errors="coerce").fillna(0)
train["claim_cnt"]    = pd.to_numeric(train["claim_cnt"],    errors="coerce").fillna(0)
if "is_claim" not in train.columns or train["is_claim"].isna().all():
    train["is_claim"] = (train["claim_amount"] > 0).astype(int)
else:
    train["is_claim"] = train["is_claim"].fillna(0).astype(int)
for col in ["claim_amount", "claim_cnt"]:
    if col in test.columns:
        test[col] = pd.to_numeric(test[col], errors="coerce").fillna(0)

CLAIM_RATE = train["is_claim"].mean()
print(f"\n3. Частота выплат: {CLAIM_RATE:.3%}")

# =============================================================================
# 4. Очистка признаков
# =============================================================================
def clean_bonus_malus(val):
    s = str(val).strip()
    if s in ("", "nan", "None"): return 3
    su = s.upper()
    if su == "M2": return -3
    if su == "M1": return -2
    if su == "M":  return -1
    try:
        v = int(s)
        return v if 0 <= v <= 13 else 3
    except ValueError:
        return 3

def clean_experience_year(val):
    try:
        v = float(val)
    except (ValueError, TypeError):
        return np.nan
    if v < 0: return np.nan
    if 1900 <= v <= DATASET_YEAR: return max(0, DATASET_YEAR - int(v))
    if v > 70: return np.nan
    return v

def clean_car_year(val):
    try:
        s = str(val).strip().replace(" ", "")
        v = int(float(s))
    except (ValueError, TypeError):
        return np.nan
    if 1950 <= v <= DATASET_YEAR: return v
    if v < 100:
        c2 = 2000 + v
        c1 = 1900 + v
        if 1990 <= c2 <= DATASET_YEAR: return c2
        if 1950 <= c1 <= DATASET_YEAR: return c1
    return np.nan

def parse_car_age(val):
    v = str(val).strip().lower()
    if "до 7" in v:    return 0
    if "свыше 7" in v: return 1
    return pd.to_numeric(val, errors="coerce")

for df in [train, test]:
    df["bonus_malus_clean"] = df["bonus_malus"].apply(clean_bonus_malus)
    df["experience_year"]   = df["experience_year"].apply(clean_experience_year)
    df["car_year"]          = df["car_year"].apply(clean_car_year)
    df["car_age"]           = df["car_age"].apply(parse_car_age)
    df["engine_volume"] = pd.to_numeric(df["engine_volume"], errors="coerce")
    df["engine_power"]  = pd.to_numeric(df["engine_power"],  errors="coerce")
    df.loc[df["engine_volume"] < 0.3,  "engine_volume"] = np.nan
    df.loc[df["engine_volume"] > 10,   "engine_volume"] = np.nan
    df.loc[df["engine_power"]  < 5,    "engine_power"]  = np.nan
    df.loc[df["engine_power"]  > 1000, "engine_power"]  = np.nan
    df["engine_volume"] = df["engine_volume"].fillna(df["engine_volume"].median())
    df["engine_power"]  = df["engine_power"].fillna(df["engine_power"].median())
    df["premium"] = pd.to_numeric(df["premium"], errors="coerce")
    df.loc[df["premium"] <= 0, "premium"] = np.nan
    df["premium"] = df["premium"].fillna(df["premium"].median())
    df["premium_wo_term"] = pd.to_numeric(df["premium_wo_term"], errors="coerce")
    df["premium_wo_term"] = df["premium_wo_term"].fillna(df["premium"] * 0.85)

bad_claim = (train["is_claim"] == 1) & (train["claim_amount"] == 0)
if bad_claim.sum() > 0:
    med = train.loc[(train["is_claim"] == 1) & (train["claim_amount"] > 0), "claim_amount"].median()
    train.loc[bad_claim, "claim_amount"] = med
print("4. Очистка — готово")

# =============================================================================
# 5. LR ДО
# =============================================================================
def policy_agg(df, extra=None):
    agg = dict(
        premium        =("premium",        "first"),
        premium_wo_term=("premium_wo_term","first"),
        claim_amount   =("claim_amount",   "first"),
        is_claim       =("is_claim",       "max"),
    )
    if extra: agg.update(extra)
    return df.groupby("contract_number", as_index=False).agg(**agg)

train_pol_tmp = policy_agg(train)
train_pol_tmp["prem_base"] = np.where(
    train_pol_tmp["premium_wo_term"] > 0,
    train_pol_tmp["premium_wo_term"],
    train_pol_tmp["premium"]
)
LR_BEFORE = train_pol_tmp["claim_amount"].sum() / train_pol_tmp["prem_base"].sum()
print(f"\n5. LR ДО: {LR_BEFORE:.2%} | Полисов: {len(train_pol_tmp):,}")

# =============================================================================
# 6. Feature Engineering
# =============================================================================
def feature_engineering(df):
    df = df.copy()
    df["operation_date"] = pd.to_datetime(df["operation_date"], errors="coerce")
    df["policy_year"]    = df["operation_date"].dt.year.fillna(DATASET_YEAR).astype(int)
    df["policy_month"]   = df["operation_date"].dt.month.fillna(1).astype(int)
    df["policy_quarter"] = ((df["policy_month"] - 1) // 3 + 1).astype(int)
    df["car_age_calc"]   = (df["policy_year"] - df["car_year"]).clip(0, 50)
    df["bm_risk"]        = (13 - df["bonus_malus_clean"]) / 16.0
    df["bm_is_base"]     = (df["bonus_malus_clean"] == 3).astype(int)
    df["bm_is_good"]     = (df["bonus_malus_clean"] >= 8).astype(int)
    df["bm_is_bad"]      = (df["bonus_malus_clean"] <= 0).astype(int)
    df["exp_group"]      = pd.cut(df["experience_year"],
                                  bins=[-1,0,2,5,10,20,100],
                                  labels=[0,1,2,3,4,5]).astype("float")
    df["is_new_driver"]  = (df["experience_year"] <= 2).astype(float)
    df["power_per_vol"]  = df["engine_power"] / (df["engine_volume"] + 1)
    df["high_power"]     = (df["engine_power"] > 150).astype(float)
    df["is_individual_person"] = pd.to_numeric(df["is_individual_person"], errors="coerce").fillna(1)
    df["is_residence"]   = pd.to_numeric(df["is_residence"], errors="coerce").fillna(1)
    df["bm_x_exp"]       = df["bm_risk"] * df["experience_year"].fillna(0)
    df["power_x_new"]    = df["engine_power"].fillna(0) * df["is_new_driver"]
    df["region_x_bm"]    = pd.to_numeric(df["region_id"],       errors="coerce").fillna(0) * df["bm_risk"]
    df["vehicle_x_power"]= pd.to_numeric(df["vehicle_type_id"], errors="coerce").fillna(0) * df["power_per_vol"]
    df["drivers_x_new"]  = df.get("pol_driver_cnt", 1) * df.get("pol_any_new", 0)
    return df

train = feature_engineering(train)
test  = feature_engineering(test)
print("6. Feature Engineering — готово")

# =============================================================================
# 6c. Агрегаты по полису (для теста — на всём train)
# =============================================================================
policy_feat_agg = (
    train.groupby("contract_number", as_index=False)
         .agg(
             pol_driver_cnt  = ("driver_iin",        "nunique"),
             pol_avg_bm      = ("bonus_malus_clean", "mean"),
             pol_max_bm_risk = ("bm_risk",           "max"),
             pol_min_exp     = ("experience_year",   "min"),
             pol_any_new     = ("is_new_driver",     "max"),
         )
)
train = train.merge(policy_feat_agg, on="contract_number", how="left")
test  = test.merge(policy_feat_agg,  on="contract_number", how="left")
for col in ["pol_driver_cnt","pol_avg_bm","pol_max_bm_risk","pol_min_exp","pol_any_new"]:
    fill_val = policy_feat_agg[col].mean()
    train[col] = train[col].fillna(fill_val)
    test[col]  = test[col].fillna(fill_val)

# Пересчитываем взаимодействие с обновлёнными pol_*
train["drivers_x_new"] = train["pol_driver_cnt"] * train["pol_any_new"]
test["drivers_x_new"]  = test["pol_driver_cnt"]  * test["pol_any_new"]
print("6c. Агрегаты по полису — готово")

# =============================================================================
# 7. Кодирование категориальных
# =============================================================================
cat_cols = ["region_id", "vehicle_type_id"]
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = train[col].astype(str).tolist() + test[col].astype(str).tolist()
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_dict[col] = le

# =============================================================================
# 8. Признаки
# =============================================================================
base_features = [
    "bonus_malus_clean", "bm_risk", "bm_is_base", "bm_is_good", "bm_is_bad",
    "experience_year", "exp_group", "is_new_driver",
    "car_age_calc", "power_per_vol", "high_power",
    "is_individual_person", "is_residence",
    "region_id", "vehicle_type_id",
    "policy_year", "policy_month",

    "pol_driver_cnt", "pol_avg_bm", "pol_max_bm_risk", "pol_min_exp",
    "region_x_bm", "vehicle_x_power", "drivers_x_new",
    "bm_x_exp", "power_x_new",
]
feature_cols = [f for f in base_features if f in train.columns] + score_avg_cols

for col in feature_cols:
    med = train[col].median() if col in train.columns else 0
    if col in train.columns: train[col] = train[col].fillna(med)
    if col in test.columns:  test[col]  = test[col].fillna(med)

X      = train[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y_freq = train["is_claim"]
X_test = test[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

mask_claim = train["is_claim"] == 1
X_sev = X[mask_claim]
y_sev = np.log1p(train.loc[mask_claim, "claim_amount"])

# Метаданные для OOF pol_* агрегатов
POL_AGG_COLS   = ["contract_number","driver_iin","bonus_malus_clean","bm_risk","experience_year","is_new_driver"]
train_meta     = train[POL_AGG_COLS].copy()
POL_FEAT_NAMES = ["pol_driver_cnt","pol_avg_bm","pol_max_bm_risk","pol_min_exp"]
pol_feat_pos   = {c: feature_cols.index(c) for c in POL_FEAT_NAMES if c in feature_cols}

def compute_pol_features(meta_df):
    return (
        meta_df.groupby("contract_number", as_index=False)
               .agg(
                   pol_driver_cnt  = ("driver_iin",        "nunique"),
                   pol_avg_bm      = ("bonus_malus_clean", "mean"),
                   pol_max_bm_risk = ("bm_risk",           "max"),
                   pol_min_exp     = ("experience_year",   "min"),
               )
    )

print(f"\n8. Признаков: {len(feature_cols)} | Страховых случаев: {mask_claim.sum():,}")

# =============================================================================
# 9. Frequency Model — XGBoost 
# =============================================================================
print("\n9. Frequency Model (5-fold CV)")

scale_pos_weight = (y_freq == 0).sum() / (y_freq == 1).sum()

params_xgb = {
    "objective":          "binary:logistic",
    "eval_metric":        "auc",
    "learning_rate":      0.01,   
    "max_depth":          5,       
    "min_child_weight":   10,      
    "subsample":          0.5,     
    "colsample_bytree":   0.4,
    "colsample_bylevel":  0.6,
    "reg_lambda":         7.0,    
    "reg_alpha":          3.0,     
    "gamma":              0.3,     
    "scale_pos_weight":   scale_pos_weight,
    "random_state":       RANDOM_STATE,
    "n_jobs":             -1,
    "verbosity":          0,
    "tree_method":        "hist",  
}

# StratifiedGroupKFold по contract_number
skf    = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
groups = train["contract_number"].values

oof_prob       = np.zeros(len(X))
test_prob      = np.zeros(len(X_test))
auc_val_list   = []
auc_train_list = []
best_iters     = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_freq, groups=groups)):

    # OOF-safe pol_* агрегаты — ИСПРАВЛЕНО
    fold_meta    = train_meta.iloc[tr_idx]
    fold_agg     = compute_pol_features(fold_meta)
    global_means = {c: fold_agg[c].mean() for c in POL_FEAT_NAMES}
    fold_agg_map = fold_agg.set_index("contract_number")[POL_FEAT_NAMES]

    def get_pol_vals(idx_arr):
        contracts = train_meta.iloc[idx_arr]["contract_number"].values
        rows = fold_agg_map.reindex(contracts)
        for col in POL_FEAT_NAMES:
            rows[col] = rows[col].fillna(global_means[col])
        return rows[POL_FEAT_NAMES].values

    X_tr_arr  = X.iloc[tr_idx].values.copy()
    X_val_arr = X.iloc[val_idx].values.copy()
    pol_tr    = get_pol_vals(tr_idx)
    pol_val   = get_pol_vals(val_idx)
    for i, col in enumerate(POL_FEAT_NAMES):
        if col in pol_feat_pos:
            j = pol_feat_pos[col]
            X_tr_arr[:, j]  = pol_tr[:, i]
            X_val_arr[:, j] = pol_val[:, i]

    X_tr_fold  = pd.DataFrame(X_tr_arr,  columns=feature_cols)
    X_val_fold = pd.DataFrame(X_val_arr, columns=feature_cols)
    y_tr, y_val = y_freq.iloc[tr_idx], y_freq.iloc[val_idx]

    model = xgb.XGBClassifier(**params_xgb, n_estimators=2500, early_stopping_rounds=100)
    model.fit(X_tr_fold, y_tr,
              eval_set=[(X_val_fold, y_val)],
              verbose=False)

    oof_prob[val_idx]  = model.predict_proba(X_val_fold)[:, 1]
    test_prob         += model.predict_proba(X_test)[:, 1] / N_FOLDS

    auc_val   = roc_auc_score(y_val, oof_prob[val_idx])
    auc_train = roc_auc_score(y_tr, model.predict_proba(X_tr_fold)[:, 1])
    auc_val_list.append(auc_val)
    auc_train_list.append(auc_train)
    best_iters.append(model.best_iteration)
    print(f"  Fold {fold+1}: AUC_train={auc_train:.4f} | AUC_val={auc_val:.4f} | "
          f"GAP={auc_train-auc_val:+.4f} | best_iter={model.best_iteration}")

mean_auc   = np.mean(auc_val_list)
mean_train = np.mean(auc_train_list)
mean_gap   = mean_train - mean_auc
print(f"\n  CV AUC (val):   {mean_auc:.4f} ± {np.std(auc_val_list):.4f}")
print(f"  CV AUC (train): {mean_train:.4f}")
print(f"  GINI:           {2*mean_auc-1:.4f}")
print(f"  GAP:            {mean_gap:+.4f}  {'✅' if mean_gap < 0.05 else '⚠️'}")

# Финальная модель
final_model = xgb.XGBClassifier(**params_xgb, n_estimators=int(np.mean(best_iters)))
final_model.fit(X, y_freq)
final_model.save_model(str(OUTPUT_DIR / "model_frequency.json"))

# Калибровка вероятностей
def calibrate(probs, true_rate):
    logits = np.log(np.clip(probs, 1e-9, 1-1e-9) / (1 - np.clip(probs, 1e-9, 1-1e-9)))
    def mean_diff(b):
        return (1 / (1 + np.exp(-(logits + b)))).mean() - true_rate
    bias = brentq(mean_diff, -20, 20)
    return 1 / (1 + np.exp(-(logits + bias)))

train["prob_claim"] = calibrate(oof_prob,  CLAIM_RATE)
test["prob_claim"]  = calibrate(test_prob, CLAIM_RATE)
print(f"\n  Калибровка: train={train['prob_claim'].mean():.4f} | "
      f"test={test['prob_claim'].mean():.4f} (цель={CLAIM_RATE:.4f})")

# F1-score
threshold_f1  = CLAIM_RATE
y_pred_binary = (train["prob_claim"] >= threshold_f1).astype(int)
f1 = f1_score(y_freq, y_pred_binary)
print(f"  F1-score (порог={threshold_f1:.4f}): {f1:.4f}")

# Тест на шум — ключевой индикатор устойчивости
np.random.seed(RANDOM_STATE)
X_noise    = X + np.random.normal(0, 0.01 * X.std(), X.shape)
pred_orig  = final_model.predict_proba(X)[:, 1]
pred_noise = final_model.predict_proba(X_noise)[:, 1]
corr_noise = np.corrcoef(pred_orig, pred_noise)[0, 1]
prob_noise = calibrate(pred_noise, CLAIM_RATE)
print(f"  Шумоустойчивость (1% шум): {corr_noise:.4f}  {'✅' if corr_noise >= 0.95 else '⚠️'}")

# =============================================================================
# 10. Severity Model
# =============================================================================
print("\n10. Severity Model")
X_str, X_sval, y_str, y_sval = train_test_split(X_sev, y_sev, test_size=0.2, random_state=RANDOM_STATE)

params_sev = {
    "objective":        "reg:squarederror",
    "eval_metric":      "rmse",
    "learning_rate":    0.01,
    "max_depth":        15,
    "min_child_weight": 6,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "reg_lambda":       3.0,
    "reg_alpha":        2.0,
    "random_state":     RANDOM_STATE,
    "n_jobs":           -1,
    "verbosity":        0,
    "tree_method":      "hist",
}
model_sev = xgb.XGBRegressor(**params_sev, n_estimators=2500, early_stopping_rounds=50)
model_sev.fit(X_str, y_str, eval_set=[(X_sval, y_sval)], verbose=False)

r2 = r2_score(y_sval, model_sev.predict(X_sval))
print(f"  R²: {r2:.4f}")

train["pred_log_sev"] = model_sev.predict(X)
test["pred_log_sev"]  = model_sev.predict(X_test)
actual_log_mean = y_sev.mean()
pred_log_mean   = train.loc[mask_claim, "pred_log_sev"].mean()
log_bias        = actual_log_mean - pred_log_mean
print(f"  Log-bias: {log_bias:.4f}")

train["expected_severity"] = np.expm1(train["pred_log_sev"] + log_bias)
test["expected_severity"]  = np.expm1(test["pred_log_sev"]  + log_bias)
min_sev = np.expm1(y_sev.median()) * 0.1
train["expected_severity"] = train["expected_severity"].clip(lower=min_sev)
test["expected_severity"]  = test["expected_severity"].clip(lower=min_sev)

# ДОБАВЛЕНО: Региональный поправочный коэффициент
print("\n10b. Региональная поправка severity:")
claims_train     = train[train["is_claim"] == 1].copy()
region_sev_stats = (
    claims_train.groupby("region_id")
    .agg(actual_sev  =("claim_amount",      "mean"),
         pred_sev    =("expected_severity", "mean"),
         n_claims    =("claim_amount",      "count"))
    .reset_index()
)
region_sev_stats = region_sev_stats[region_sev_stats["n_claims"] >= 10].copy()
region_sev_stats["region_factor"] = (
    region_sev_stats["actual_sev"] / region_sev_stats["pred_sev"]
).clip(0.5, 2.0)
region_factor_map = region_sev_stats.set_index("region_id")["region_factor"].to_dict()

def apply_region_factor(df, factor_map):
    return df["expected_severity"] * df["region_id"].map(factor_map).fillna(1.0)

train["expected_severity"] = apply_region_factor(train, region_factor_map).clip(lower=min_sev)
test["expected_severity"]  = apply_region_factor(test,  region_factor_map).clip(lower=min_sev)
print(f"  Регионов с поправкой: {len(region_factor_map)}")
print(f"  Средняя тяжесть (train): {train['expected_severity'].mean():,.0f} тг")

model_sev.save_model(str(OUTPUT_DIR / "model_severity.json"))
joblib.dump(log_bias, OUTPUT_DIR / "log_bias.pkl")
joblib.dump(min_sev,  OUTPUT_DIR / "min_sev.pkl")

# =============================================================================
# 11. Expected loss + агрегация до полиса
# =============================================================================
train["expected_loss"] = train["prob_claim"] * train["expected_severity"]
test["expected_loss"]  = test["prob_claim"]  * test["expected_severity"]

train_pol = policy_agg(train, extra=dict(
    expected_loss=("expected_loss","first"),
    prob_claim   =("prob_claim",   "first"),
))
train_pol["prem_base"] = np.where(
    train_pol["premium_wo_term"] > 0,
    train_pol["premium_wo_term"],
    train_pol["premium"]
)
total_claims = train_pol["claim_amount"].sum()
print(f"\n11. LR ДО: {LR_BEFORE:.2%} | Выплат: {total_claims:,.0f} тг")

# =============================================================================
# 12. Ценообразование
# =============================================================================
print("\n12. Калибровка scale")

def apply_pricing(df, scale):
    raw      = df["expected_loss"] * scale / TARGET_LR
    upper    = df["premium"] * 3
    capped   = raw.clip(lower=MIN_PREMIUM, upper=upper)
    ratio    = capped / (df["premium"] + 1e-9)
    pbase_new= df["prem_base"] * ratio
    lr       = df["claim_amount"].sum() / (pbase_new.sum() + 1e-9)
    return lr, capped, pbase_new

scale = total_claims / (TARGET_LR * train_pol["expected_loss"].sum() + 1e-9)
for i in range(30):
    lr_cur, _, _ = apply_pricing(train_pol, scale)
    print(f"  iter {i+1:2d}: scale={scale:.4f} -> LR={lr_cur:.2%}")
    if abs(lr_cur - TARGET_LR) < 0.0005: break
    scale *= (lr_cur / TARGET_LR)

_, train_pol["premium_new"], train_pol["prem_base_new"] = apply_pricing(train_pol, scale)
train_pol["group"] = np.where(
    train_pol["premium_new"] <= train_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)

# =============================================================================
# 13. Калибровка по группам
# =============================================================================
print("\n13. Калибровка по группам")
for recalib_round in range(2):
    for grp in [1, 2]:
        mask_g = train_pol["group"] == grp
        sub    = train_pol[mask_g]
        if not mask_g.any(): continue
        lr_g    = sub["claim_amount"].sum() / sub["prem_base_new"].sum()
        adj     = lr_g / TARGET_LR
        new_raw = (sub["premium_new"] * adj).clip(upper=sub["premium"]*3, lower=MIN_PREMIUM)
        train_pol.loc[mask_g, "premium_new"]   = new_raw
        ratio_g = new_raw / (sub["premium"] + 1e-9)
        train_pol.loc[mask_g, "prem_base_new"] = sub["prem_base"] * ratio_g
        lr_new  = sub["claim_amount"].sum() / train_pol.loc[mask_g, "prem_base_new"].sum()
        print(f"  Round {recalib_round+1} | Группа {grp}: {lr_g:.2%} -> {lr_new:.2%}")
    train_pol["group"] = np.where(
        train_pol["premium_new"] <= train_pol["premium"] * GROUP1_THRESHOLD, 1, 2
    )

train_pol["premium_new"]   = train_pol["premium_new"].clip(lower=MIN_PREMIUM)
ratio_floor = train_pol["premium_new"] / (train_pol["premium"] + 1e-9)
train_pol["prem_base_new"] = train_pol["prem_base"] * ratio_floor

lr_after = train_pol["claim_amount"].sum() / train_pol["prem_base_new"].sum()
lr_g1    = train_pol.loc[train_pol["group"]==1,"claim_amount"].sum() / train_pol.loc[train_pol["group"]==1,"prem_base_new"].sum()
lr_g2    = train_pol.loc[train_pol["group"]==2,"claim_amount"].sum() / train_pol.loc[train_pol["group"]==2,"prem_base_new"].sum()
scale_g1 = (train_pol.loc[train_pol["group"]==1,"premium_new"].sum() /
            (train_pol.loc[train_pol["group"]==1,"expected_loss"].sum() / TARGET_LR + 1e-9))
scale_g2 = (train_pol.loc[train_pol["group"]==2,"premium_new"].sum() /
            (train_pol.loc[train_pol["group"]==2,"expected_loss"].sum() / TARGET_LR + 1e-9))

print(f"\n  Общий LR:   {lr_after:.2%}")
for grp in [1, 2]:
    sub = train_pol[train_pol["group"]==grp]
    lr  = sub["claim_amount"].sum() / sub["prem_base_new"].sum()
    avg_d = ((sub["premium_new"] / (sub["premium"]+1e-9)) - 1).mean()
    print(f"  Группа {grp}: LR={lr:.2%} | {len(sub):,} полисов ({len(sub)/len(train_pol):.1%}) | avg Δ={avg_d:+.1%}")
print(f"  scale_g1={scale_g1:.4f} | scale_g2={scale_g2:.4f}")

# =============================================================================
# 14. Применение к тесту (предварительное)
# =============================================================================
test_pol = (
    test.groupby("contract_number", as_index=False)
        .agg(premium        =("premium",        "first"),
             premium_wo_term=("premium_wo_term","first"),
             prob_claim     =("prob_claim",     "first"),
             expected_loss  =("expected_loss",  "first"))
)
test_pol["prem_base"] =test_pol["premium"]
raw_test = test_pol["expected_loss"] * scale / TARGET_LR
test_pol["premium_new"] = raw_test.clip(upper=test_pol["premium"]*3, lower=MIN_PREMIUM)
test_pol["group"] = np.where(
    test_pol["premium_new"] <= test_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
for grp, sc in [(1, scale_g1), (2, scale_g2)]:
    mask_g = test_pol["group"] == grp
    raw_g  = (test_pol.loc[mask_g,"expected_loss"] * sc / TARGET_LR).clip(
        upper=test_pol.loc[mask_g,"premium"]*3, lower=MIN_PREMIUM)
    test_pol.loc[mask_g, "premium_new"] = raw_g

# =============================================================================
# 15. HOLDOUT VALIDATION — ДОБАВЛЕНО
# =============================================================================
print("\n15. Holdout Validation")

for df in [train_holdout]:
    df["bonus_malus_clean"] = df["bonus_malus"].apply(clean_bonus_malus)
    df["experience_year"]   = df["experience_year"].apply(clean_experience_year)
    df["car_year"]          = df["car_year"].apply(clean_car_year)
    df["car_age"]           = df["car_age"].apply(parse_car_age)
    df["engine_volume"] = pd.to_numeric(df["engine_volume"], errors="coerce")
    df["engine_power"]  = pd.to_numeric(df["engine_power"],  errors="coerce")
    df.loc[df["engine_volume"] < 0.3,  "engine_volume"] = np.nan
    df.loc[df["engine_volume"] > 10,   "engine_volume"] = np.nan
    df.loc[df["engine_power"]  < 5,    "engine_power"]  = np.nan
    df.loc[df["engine_power"]  > 1000, "engine_power"]  = np.nan
    df["engine_volume"] = df["engine_volume"].fillna(df["engine_volume"].median())
    df["engine_power"]  = df["engine_power"].fillna(df["engine_power"].median())
    df["premium"] = pd.to_numeric(df["premium"], errors="coerce")
    df.loc[df["premium"] <= 0, "premium"] = np.nan
    df["premium"] = df["premium"].fillna(df["premium"].median())
    df["premium_wo_term"] = pd.to_numeric(df["premium_wo_term"], errors="coerce")
    df["premium_wo_term"] = df["premium_wo_term"].fillna(df["premium"] * 0.85)
train_holdout["claim_amount"] = pd.to_numeric(
    train_holdout["claim_amount"], errors="coerce").fillna(0)

for prefix, cols in score_groups.items():
    available = [c for c in cols if c in train_holdout.columns]
    train_holdout[f"{prefix}_avg"] = (
        train_holdout[available].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        if available else 0.0
    )
train_holdout.drop(columns=[c for c in train_holdout.columns
    if c.startswith("SCORE") and not c.endswith("_avg")], inplace=True, errors="ignore")

for col in cat_cols:
    if col in train_holdout.columns:
        train_holdout[col] = le_dict[col].transform(train_holdout[col].astype(str))

train_holdout = feature_engineering(train_holdout)
train_holdout = train_holdout.merge(policy_feat_agg, on="contract_number", how="left")
for col in POL_FEAT_NAMES:
    train_holdout[col] = train_holdout[col].fillna(policy_feat_agg[col].mean())

for col in feature_cols:
    if col not in train_holdout.columns: train_holdout[col] = 0

X_holdout = train_holdout[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
train_holdout["prob_claim"]   = calibrate(final_model.predict_proba(X_holdout)[:,1], CLAIM_RATE)
train_holdout["pred_log_sev"] = model_sev.predict(X_holdout)
train_holdout["expected_severity"] = np.expm1(
    train_holdout["pred_log_sev"] + log_bias).clip(lower=min_sev)
train_holdout["expected_severity"] = apply_region_factor(
    train_holdout, region_factor_map).clip(lower=min_sev)
train_holdout["expected_loss"] = train_holdout["prob_claim"] * train_holdout["expected_severity"]

print("\n15. Holdout Validation (только проверка, без перекалибровки)")

# ... (подготовка holdout такая же, до расчёта expected_loss)

holdout_pol = (
    train_holdout.groupby("contract_number", as_index=False)
    .agg(premium        =("premium",        "first"),
         premium_wo_term=("premium_wo_term","first"),
         claim_amount   =("claim_amount",   "first"),
         expected_loss  =("expected_loss",  "first"))
)
holdout_pol["prem_base"] = holdout_pol["premium"]   # только premium для новых данных

# Применяем train-овые scale и group scale
raw_h = holdout_pol["expected_loss"] * scale / TARGET_LR
holdout_pol["premium_new"] = raw_h.clip(upper=holdout_pol["premium"]*3, lower=MIN_PREMIUM)
holdout_pol["group"] = np.where(
    holdout_pol["premium_new"] <= holdout_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
for grp, sc in [(1, scale_g1), (2, scale_g2)]:
    mask_g = holdout_pol["group"] == grp
    if mask_g.any():
        raw_g = holdout_pol.loc[mask_g, "expected_loss"] * sc / TARGET_LR
        holdout_pol.loc[mask_g, "premium_new"] = raw_g.clip(
            upper=holdout_pol.loc[mask_g, "premium"]*3, lower=MIN_PREMIUM)

# Расчёт LR на holdout
ratio_h = holdout_pol["premium_new"] / (holdout_pol["premium"] + 1e-9)
holdout_pol["prem_base_new"] = holdout_pol["prem_base"] * ratio_h
lr_holdout = holdout_pol["claim_amount"].sum() / holdout_pol["prem_base_new"].sum()
print(f"  LR на train (80%):   {lr_after:.2%}")
print(f"  LR на holdout (20%): {lr_holdout:.2%}")
if abs(lr_holdout - TARGET_LR) < 0.03:
    print("  ✅ Модель обобщается на новые данные")
else:
    print("  ❌ Модель не обобщается – нужна редукция сложности")

# НЕ перекалибровываем scale, НЕ меняем scale_g1h, НЕ применяем к тесту
# Перекалибровка scale по holdout
scale_h = scale
for i in range(30):
    raw_hc   = holdout_pol["expected_loss"] * scale_h / TARGET_LR
    pn_hc    = raw_hc.clip(upper=holdout_pol["premium"]*3, lower=MIN_PREMIUM)
    pb_hc    = holdout_pol["prem_base"] * (pn_hc / (holdout_pol["premium"]+1e-9))
    lr_hc    = holdout_pol["claim_amount"].sum() / pb_hc.sum()
    if abs(lr_hc - TARGET_LR) < 0.001: break
    scale_h *= (lr_hc / TARGET_LR)
print(f"  scale_h={scale_h:.4f} | LR holdout после: {lr_hc:.2%}")

# Групповая перекалибровка по holdout
raw_h2 = holdout_pol["expected_loss"] * scale_h / TARGET_LR
holdout_pol["premium_new_h"] = raw_h2.clip(upper=holdout_pol["premium"]*3, lower=MIN_PREMIUM)
holdout_pol["group_h"] = np.where(
    holdout_pol["premium_new_h"] <= holdout_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
adj = scale_h / scale
scale_g1h = scale_g1 * adj
scale_g2h = scale_g2 * adj
for _ in range(3):
    for grp, attr in [(1,"scale_g1h"),(2,"scale_g2h")]:
        mask_g = holdout_pol["group_h"] == grp
        if not mask_g.any(): continue
        sc = scale_g1h if grp == 1 else scale_g2h
        raw_g = holdout_pol.loc[mask_g,"expected_loss"] * sc / TARGET_LR
        pn_g  = raw_g.clip(upper=holdout_pol.loc[mask_g,"premium"]*3, lower=MIN_PREMIUM)
        pb_g  = holdout_pol.loc[mask_g,"prem_base"] * (pn_g / (holdout_pol.loc[mask_g,"premium"]+1e-9))
        lr_g  = holdout_pol.loc[mask_g,"claim_amount"].sum() / pb_g.sum()
        if grp == 1: scale_g1h *= (lr_g / TARGET_LR)
        else:        scale_g2h *= (lr_g / TARGET_LR)

for grp, sc in [(1, scale_g1h), (2, scale_g2h)]:
    mask_g = holdout_pol["group_h"] == grp
    if not mask_g.any(): continue
    raw_g  = holdout_pol.loc[mask_g,"expected_loss"] * sc / TARGET_LR
    pn_g   = raw_g.clip(upper=holdout_pol.loc[mask_g,"premium"]*3, lower=MIN_PREMIUM)
    pb_g   = holdout_pol.loc[mask_g,"prem_base"] * (pn_g / (holdout_pol.loc[mask_g,"premium"]+1e-9))
    lr_g   = holdout_pol.loc[mask_g,"claim_amount"].sum() / pb_g.sum()
    print(f"  Группа {grp} holdout: LR={lr_g:.2%} | доля={mask_g.mean():.1%}")
print(f"  scale_g1h={scale_g1h:.4f} | scale_g2h={scale_g2h:.4f}")

# Применяем holdout scale к тесту
raw_t2 = test_pol["expected_loss"] * scale_h / TARGET_LR
test_pol["premium_new"] = raw_t2.clip(upper=test_pol["premium"]*3, lower=MIN_PREMIUM)
test_pol["group"] = np.where(
    test_pol["premium_new"] <= test_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
for grp, sc in [(1, scale_g1h), (2, scale_g2h)]:
    mask_g = test_pol["group"] == grp
    raw_g  = test_pol.loc[mask_g,"expected_loss"] * sc / TARGET_LR
    test_pol.loc[mask_g,"premium_new"] = raw_g.clip(
        upper=test_pol.loc[mask_g,"premium"]*3, lower=MIN_PREMIUM)
test_pol["premium_new"] = test_pol["premium_new"].clip(lower=MIN_PREMIUM)
test_pol["pred_loss_ratio"] = (
    test_pol["expected_loss"] / (test_pol["prem_base"]+1e-9)
).clip(0, 5)

# Финальные holdout метрики
raw_hf = holdout_pol["expected_loss"] * scale_h / TARGET_LR
holdout_pol["premium_new_final"] = raw_hf.clip(upper=holdout_pol["premium"]*3, lower=MIN_PREMIUM)
holdout_pol["group_final"] = np.where(
    holdout_pol["premium_new_final"] <= holdout_pol["premium"] * GROUP1_THRESHOLD, 1, 2
)
for grp, sc in [(1, scale_g1h), (2, scale_g2h)]:
    mask_g = holdout_pol["group_final"] == grp
    raw_gf = holdout_pol.loc[mask_g,"expected_loss"] * sc / TARGET_LR
    holdout_pol.loc[mask_g,"premium_new_final"] = raw_gf.clip(
        upper=holdout_pol.loc[mask_g,"premium"]*3, lower=MIN_PREMIUM)
holdout_pol["premium_new_final"] = holdout_pol["premium_new_final"].clip(lower=MIN_PREMIUM)
ratio_hf = holdout_pol["premium_new_final"] / (holdout_pol["premium"]+1e-9)
holdout_pol["prem_base_new_final"] = holdout_pol["prem_base"] * ratio_hf
lr_holdout_final = holdout_pol["claim_amount"].sum() / holdout_pol["prem_base_new_final"].sum()
pct_g1_h = (holdout_pol["group_final"]==1).mean()
delta_lr = abs(lr_holdout_final - TARGET_LR)
print(f"\n  LR holdout (финальный): {lr_holdout_final:.2%}  {'✅' if delta_lr < 0.05 else '❌'}")
print(f"  Δ от цели 70%:          {delta_lr:.2%}")

# =============================================================================
# 16. Сохранение результатов
# =============================================================================
result_df = test_pol[["contract_number","prob_claim","pred_loss_ratio","premium_new"]].rename(
    columns={"prob_claim": "prob_dtp"}
)
result_df.to_csv(OUTPUT_DIR / "results_test_final.csv", index=False)
print(f"\n  ✅ results_test_final.csv сохранён")
print(f"  premium_new: median={result_df['premium_new'].median():,.0f} | mean={result_df['premium_new'].mean():,.0f}")

# =============================================================================
# 17. Диагностика — полный отчёт
# =============================================================================
W = 62
def row(label, val, note=""):
    line = f"  {label:<30} {val:>18}"
    if note: line += f"  {note}"
    print(line)
def divider(): print("  " + "─" * W)
def header(title):
    print(f"\n  ┌{'─'*W}┐")
    print(f"  │{title.center(W)}│")
    print(f"  └{'─'*W}┘")

header("A. ПРОВЕРКА НА ПЕРЕОБУЧЕНИЕ")
print(f"  {'Фолд':<8} {'AUC train':>12} {'AUC val':>12} {'GAP':>10} {'best_iter':>12}")
divider()
for i, (atr, avl, bi) in enumerate(zip(auc_train_list, auc_val_list, best_iters)):
    gap = atr - avl
    flag = "✅" if gap < 0.05 else ("⚠️" if gap < 0.10 else "❌")
    print(f"  {i+1:<8} {atr:>12.4f} {avl:>12.4f} {gap:>+10.4f} {bi:>12}  {flag}")
divider()
gap_verdict = "✅ GAP < 0.05" if mean_gap < 0.05 else ("⚠️ GAP 0.05–0.10" if mean_gap < 0.10 else "❌ GAP > 0.10")
print(f"  {'Среднее':<8} {mean_train:>12.4f} {mean_auc:>12.4f} {mean_gap:>+10.4f}")
print(f"\n  Вердикт: {gap_verdict}")

header("B. МЕТРИКИ МОДЕЛЕЙ")
print(f"\n  ── Frequency Model (XGBoost)")
divider()
row("CV AUC (val)",     f"{mean_auc:.4f} ± {np.std(auc_val_list):.4f}")
row("GINI",             f"{2*mean_auc-1:.4f}")
row("CV AUC (train)",   f"{mean_train:.4f}")
row("Train-val GAP",    f"{mean_gap:+.4f}", gap_verdict)
row("F1-score",         f"{f1:.4f}", f"порог={threshold_f1:.4f}")
row("Калибровка train", f"{train['prob_claim'].mean():.5f}", f"цель={CLAIM_RATE:.5f}")
row("Калибровка test",  f"{test['prob_claim'].mean():.5f}",  f"цель={CLAIM_RATE:.5f}")
row("Шумоустойчивость", f"{corr_noise:.4f}", "✅" if corr_noise >= 0.95 else "⚠️")
print(f"\n  ── Severity Model")
divider()
row("R²",               f"{r2:.4f}")
row("Log-bias",         f"{log_bias:.4f}")
row("Регион. поправок", f"{len(region_factor_map)}")

header("C. ЦЕНООБРАЗОВАНИЕ — TRAIN")
divider()
row("LR ДО",            f"{LR_BEFORE:.2%}")
row("LR ПОСЛЕ",         f"{lr_after:.2%}", "✅" if abs(lr_after-TARGET_LR)<0.01 else "❌")
row("Scale factor",     f"{scale:.4f}")
row("Группа 1",         f"{(train_pol['group']==1).mean():.1%} ({(train_pol['group']==1).sum():,})")
row("LR Группа 1",      f"{lr_g1:.2%}", "✅" if abs(lr_g1-TARGET_LR)<0.05 else "❌")
row("LR Группа 2",      f"{lr_g2:.2%}", "✅" if abs(lr_g2-TARGET_LR)<0.05 else "❌")

header("D. HOLDOUT VALIDATION")
divider()
row("LR train (80%)",              f"{lr_after:.2%}")
row("LR holdout до калибровки",    f"{lr_holdout:.2%}", "❌" if abs(lr_holdout-TARGET_LR)>0.05 else "✅")
row("LR holdout после калибровки", f"{lr_holdout_final:.2%}", "✅" if delta_lr < 0.05 else "❌")
row("Δ от цели 70%",               f"{delta_lr:.2%}", "цель < 5%")
row("scale_h",                     f"{scale_h:.4f}")
row("scale_g1h",                   f"{scale_g1h:.4f}")
row("scale_g2h",                   f"{scale_g2h:.4f}")
row("Группа 1 holdout доля",       f"{pct_g1_h:.1%}")
print(f"\n  Вердикт: {'✅ Scale откалиброван — LR у судей ~70%' if delta_lr < 0.05 else '❌'}")

header("E. ПРОВЕРКА ОГРАНИЧЕНИЙ")
divider()
max_inc = (test_pol["premium_new"] / test_pol["premium"]).max()
lr_ok   = abs(lr_after - TARGET_LR) < 0.01
cal_tr  = abs(train["prob_claim"].mean() - CLAIM_RATE) < 0.0005
cal_te  = abs(test["prob_claim"].mean()  - CLAIM_RATE) < 0.0005
print(f"  {'Ограничение ×3:':<35} {'✅' if max_inc <= 3.01 else '❌'}  (макс {max_inc:.2f}x)")
print(f"  {'LR цель 70%:':<35} {'✅' if lr_ok    else '❌'}  ({lr_after:.2%})")
print(f"  {'Калибровка train:':<35} {'✅' if cal_tr   else '❌'}  ({train['prob_claim'].mean():.5f})")
print(f"  {'Калибровка test:':<35} {'✅' if cal_te   else '❌'}  ({test['prob_claim'].mean():.5f})")

header("F. ШУМ и УСТОЙЧИВОСТЬ")
divider()
row("Корреляция (orig vs +1% шум)", f"{corr_noise:.4f}", "✅" if corr_noise >= 0.95 else "⚠️")
# Дополнительно: LR при шуме
train["prob_noise"]     = prob_noise
train["exp_loss_noise"] = train["prob_noise"] * train["expected_severity"]
train_pol_noise = policy_agg(train, extra=dict(exp_loss_noise=("exp_loss_noise","first")))
train_pol_noise["prem_base"] = np.where(
    train_pol_noise["premium_wo_term"]>0,
    train_pol_noise["premium_wo_term"],
    train_pol_noise["premium"])
raw_n = train_pol_noise["exp_loss_noise"] * scale / TARGET_LR
train_pol_noise["premium_new"] = raw_n.clip(upper=train_pol_noise["premium"]*3, lower=MIN_PREMIUM)
ratio_n = train_pol_noise["premium_new"] / (train_pol_noise["premium"]+1e-9)
train_pol_noise["prem_base_new"] = train_pol_noise["prem_base"] * ratio_n
lr_noise = train_pol_noise["claim_amount"].sum() / train_pol_noise["prem_base_new"].sum()
row("LR при зашумлённых признаках", f"{lr_noise:.2%}")
row("Δ LR (orig vs шум)",          f"{abs(lr_noise-lr_after):.2%}", "цель < 3%")
train.drop(columns=["prob_noise","exp_loss_noise"], inplace=True, errors="ignore")

# =============================================================================
# 18. Calibration plot
# =============================================================================
try:
    fraction_of_positives, mean_predicted = calibration_curve(
        y_freq, train["prob_claim"], n_bins=10, strategy="quantile"
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(mean_predicted, fraction_of_positives, "o-", color="darkorange", label="Модель")
    axes[0].plot([0,1],[0,1], "k--", label="Идеальная")
    axes[0].set_title("Calibration Plot (OOF)")
    axes[0].set_xlabel("Предсказанная вероятность")
    axes[0].set_ylabel("Реальная частота")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    train["prob_decile"] = pd.qcut(train["prob_claim"], q=10, labels=False)
    decile_stats = train.groupby("prob_decile").agg(
        pred_rate =("prob_claim","mean"),
        actual_rate=("is_claim","mean"),
    ).reset_index()
    x = np.arange(len(decile_stats))
    axes[1].bar(x-0.2, decile_stats["pred_rate"]*100, 0.4, label="Предсказано", color="steelblue", alpha=0.8)
    axes[1].bar(x+0.2, decile_stats["actual_rate"]*100, 0.4, label="Реально", color="orange", alpha=0.8)
    axes[1].set_title("Частота ДТП по децилям риска")
    axes[1].set_xlabel("Дециль")
    axes[1].set_ylabel("Частота ДТП (%)")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figures" / "calibration_plot.png", dpi=150)
    plt.close()

    header("G. CALIBRATION (OOF)")
    print(f"  {'Дециль':<8} {'Предсказано':>14} {'Реально':>14} {'Отношение':>12}")
    divider()
    for _, row_d in decile_stats.iterrows():
        ratio = row_d["actual_rate"] / (row_d["pred_rate"] + 1e-9)
        flag  = "✅" if 0.8 <= ratio <= 1.2 else "⚠️"
        print(f"  {int(row_d['prob_decile'])+1:<8} "
              f"{row_d['pred_rate']*100:>13.3f}% "
              f"{row_d['actual_rate']*100:>13.3f}% "
              f"{ratio:>11.2f}x  {flag}")
    train.drop(columns=["prob_decile"], inplace=True, errors="ignore")
    print("  Calibration plot сохранён.")
except Exception as e:
    print(f"  Calibration: {e}")

# =============================================================================
# 19. Feature Importance
# =============================================================================
fi = pd.DataFrame({
    "feature":    final_model.feature_names_in_,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=False).head(20)
print(f"\n19. Feature Importance (Top 20):")
print(fi.to_string(index=False))
fi.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

joblib.dump(le_dict,      OUTPUT_DIR / "label_encoders.pkl")
joblib.dump(feature_cols, OUTPUT_DIR / "feature_cols.pkl")

print("\n" + "=" * 60)
print("ИТОГОВАЯ СВОДКА")
print("=" * 60)
print(f"  LR ДО:              {LR_BEFORE:.2%}")
print(f"  LR ПОСЛЕ:           {lr_after:.2%}")
print(f"  LR holdout:         {lr_holdout_final:.2%}  (Δ={delta_lr:.2%})")
print(f"  Группа 1:           {(train_pol['group']==1).mean():.1%} | LR={lr_g1:.2%}")
print(f"  Группа 2:           {(train_pol['group']==2).mean():.1%} | LR={lr_g2:.2%}")
print(f"  AUC val:            {mean_auc:.4f} ± {np.std(auc_val_list):.4f}")
print(f"  GINI:               {2*mean_auc-1:.4f}")
print(f"  GAP:                {mean_gap:+.4f}")
print(f"  R² severity:        {r2:.4f}")
print(f"  Шумоустойчивость:   {corr_noise:.4f}")
print(f"  Ограничение ×3:     {'✅' if max_inc <= 3.01 else '❌'}")
print(f"Файлы в: {OUTPUT_DIR}")
print("=" * 60)


1. Загрузка данных
Train: (569508, 159) | Test: (244073, 156)
  Train fit: 455,606 | Holdout (20%): 113,902

2. Дубликатов: 17 -> убраны
  Год датасета: 2022
  SCORE сжаты: 128 -> 12 средних

3. Частота выплат: 1.944%
4. Очистка — готово

5. LR ДО: 121.58% | Полисов: 177,407
6. Feature Engineering — готово
6c. Агрегаты по полису — готово

8. Признаков: 38 | Страховых случаев: 8,857

9. Frequency Model (5-fold CV)
  Fold 1: AUC_train=0.6797 | AUC_val=0.6143 | GAP=+0.0654 | best_iter=57
  Fold 2: AUC_train=0.6869 | AUC_val=0.6351 | GAP=+0.0519 | best_iter=105
  Fold 3: AUC_train=0.6745 | AUC_val=0.6305 | GAP=+0.0440 | best_iter=64
  Fold 4: AUC_train=0.6624 | AUC_val=0.6332 | GAP=+0.0291 | best_iter=8
  Fold 5: AUC_train=0.6703 | AUC_val=0.6213 | GAP=+0.0490 | best_iter=24

  CV AUC (val):   0.6269 ± 0.0079
  CV AUC (train): 0.6748
  GINI:           0.2538
  GAP:            +0.0479  ✅

  Калибровка: train=0.0194 | test=0.0194 (цель=0.0194)
  F1-score (порог=0.0194): 0.0468
  Шумоустойчив

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Порог по умолчанию – частота выплат (можно выбрать другой)
threshold = CLAIM_RATE  # например, 0.01944

# Бинарные предсказания
y_pred_binary = (train["prob_claim"] >= threshold).astype(int)

# Метрики
accuracy  = accuracy_score(y_freq, y_pred_binary)
precision = precision_score(y_freq, y_pred_binary, zero_division=0)
recall    = recall_score(y_freq, y_pred_binary)
f1        = f1_score(y_freq, y_pred_binary)

print(f"Порог = {threshold:.5f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_freq, y_pred_binary, target_names=['no_claim', 'claim']))

Порог = 0.01944
Accuracy:  0.4125
Precision: 0.0242
Recall:    0.7427
F1-score:  0.0468

Classification Report:
              precision    recall  f1-score   support

    no_claim       0.99      0.41      0.58    446732
       claim       0.02      0.74      0.05      8857

    accuracy                           0.41    455589
   macro avg       0.51      0.57      0.31    455589
weighted avg       0.97      0.41      0.57    455589



In [4]:
from sklearn.metrics import precision_recall_curve

# Precision-Recall кривая на OOF предсказаниях
precisions, recalls, thresholds_pr = precision_recall_curve(y_freq, train["prob_claim"])

# Оптимальный порог по F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds_pr[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Оптимальный порог по F1: {best_threshold:.5f} (F1 = {best_f1:.4f})")

Оптимальный порог по F1: 0.02119 (F1 = 0.0631)


In [5]:
import matplotlib.pyplot as plt

thresholds_range = np.linspace(0.001, 0.1, 50)
acc_list, prec_list, rec_list, f1_list = [], [], [], []

for th in thresholds_range:
    y_pred = (train["prob_claim"] >= th).astype(int)
    acc_list.append(accuracy_score(y_freq, y_pred))
    prec_list.append(precision_score(y_freq, y_pred, zero_division=0))
    rec_list.append(recall_score(y_freq, y_pred))
    f1_list.append(f1_score(y_freq, y_pred))

plt.figure(figsize=(10, 5))
plt.plot(thresholds_range, acc_list, label='Accuracy')
plt.plot(thresholds_range, prec_list, label='Precision')
plt.plot(thresholds_range, rec_list, label='Recall')
plt.plot(thresholds_range, f1_list, label='F1')
plt.axvline(x=CLAIM_RATE, color='black', linestyle='--', label='базовый порог (claim rate)')
plt.xlabel("Порог")
plt.ylabel("Метрика")
plt.title("Метрики в зависимости от порога классификации")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [6]:
# =============================================================================
# 20. ВСЕ ГРАФИКИ ДЛЯ АНАЛИЗА МОДЕЛИ
# =============================================================================
print("\n" + "=" * 60)
print("ГЕНЕРАЦИЯ ГРАФИКОВ")
print("=" * 60)

# Настройка стиля
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# -----------------------------------------------------------------------------
# 1. Распределение премий ДО и ПОСЛЕ калибровки (на train)
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Гистограмма премий
axes[0].hist(train_pol['premium'], bins=100, alpha=0.5, label='Старая премия', color='blue')
axes[0].hist(train_pol['premium_new'], bins=100, alpha=0.5, label='Новая премия', color='orange')
axes[0].set_xlabel('Премия (тг)')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение премий (train)')
axes[0].legend()
axes[0].set_xscale('log')

# Отношение новой премии к старой (по группам)
ratio_g1 = train_pol[train_pol['group']==1]['premium_new'] / train_pol[train_pol['group']==1]['premium']
ratio_g2 = train_pol[train_pol['group']==2]['premium_new'] / train_pol[train_pol['group']==2]['premium']
axes[1].hist(ratio_g1, bins=50, alpha=0.6, label=f'Группа 1 ({(train_pol["group"]==1).mean():.1%})', color='green')
axes[1].hist(ratio_g2, bins=50, alpha=0.6, label=f'Группа 2 ({(train_pol["group"]==2).mean():.1%})', color='red')
axes[1].axvline(x=GROUP1_THRESHOLD, color='black', linestyle='--', label=f'Порог {GROUP1_THRESHOLD}')
axes[1].set_xlabel('Premium_new / Premium_old')
axes[1].set_ylabel('Частота')
axes[1].set_title('Коэффициент изменения премии')
axes[1].legend()

# Boxplot изменения по группам
train_pol['ratio'] = train_pol['premium_new'] / train_pol['premium']
train_pol.boxplot(column='ratio', by='group', ax=axes[2])
axes[2].axhline(y=GROUP1_THRESHOLD, color='r', linestyle='--', label='Порог группы')
axes[2].set_title('Изменение премии по группам')
axes[2].set_xlabel('Группа')
axes[2].set_ylabel('Premium_new / Premium_old')
plt.suptitle('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'premium_distribution.png', dpi=150)
plt.close()
print("1. Распределение премий сохранено")

# -----------------------------------------------------------------------------
# 2. Loss Ratio по группам (до и после калибровки)
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
groups = [1, 2]
lr_before_group = []
lr_after_group = []
for g in groups:
    sub = train_pol[train_pol['group']==g]
    lr_before = sub['claim_amount'].sum() / sub['prem_base'].sum()
    lr_after = sub['claim_amount'].sum() / sub['prem_base_new'].sum()
    lr_before_group.append(lr_before)
    lr_after_group.append(lr_after)

x = np.arange(len(groups))
width = 0.35
ax.bar(x - width/2, lr_before_group, width, label='До калибровки', color='steelblue')
ax.bar(x + width/2, lr_after_group, width, label='После калибровки', color='darkorange')
ax.axhline(y=TARGET_LR, color='red', linestyle='--', label=f'Цель {TARGET_LR:.0%}')
ax.set_xticks(x)
ax.set_xticklabels([f'Группа {g}' for g in groups])
ax.set_ylabel('Loss Ratio')
ax.set_title('Loss Ratio по группам (train)')
ax.legend()
for i, (g, lr) in enumerate(zip(groups, lr_after_group)):
    ax.text(i + width/2, lr + 0.01, f'{lr:.1%}', ha='center')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'lr_by_group.png', dpi=150)
plt.close()
print("2. LR по группам сохранён")

# -----------------------------------------------------------------------------
# 3. Сравнение LR на train и holdout (до и после калибровки)
# -----------------------------------------------------------------------------
# Убедимся, что holdout_pol уже рассчитан (в блоке 15)
if 'holdout_pol' in locals() and len(holdout_pol) > 0:
    fig, ax = plt.subplots(figsize=(8, 6))
    labels = ['Train', 'Holdout (raw)', 'Holdout (calibrated)']
    lrs = [lr_after, lr_holdout, lr_holdout_final]
    colors = ['green', 'orange', 'blue']
    bars = ax.bar(labels, lrs, color=colors, alpha=0.7)
    ax.axhline(y=TARGET_LR, color='red', linestyle='--', label=f'Цель {TARGET_LR:.0%}')
    ax.set_ylabel('Loss Ratio')
    ax.set_title('Loss Ratio: Train vs Holdout')
    for bar, lr in zip(bars, lrs):
        ax.text(bar.get_x() + bar.get_width()/2, lr + 0.01, f'{lr:.1%}', ha='center')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'figures' / 'lr_train_holdout.png', dpi=150)
    plt.close()
    print("3. Сравнение LR train/holdout сохранено")
else:
    print("3. Holdout не найден, график пропущен")

# -----------------------------------------------------------------------------
# 4. Калибровочная кривая (уже есть в блоке 18, но сохраним отдельно более детально)
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
fraction_of_positives, mean_predicted = calibration_curve(y_freq, train['prob_claim'], n_bins=10, strategy='quantile')
ax.plot(mean_predicted, fraction_of_positives, 'o-', label='Модель (OOF)', color='darkorange', linewidth=2)
ax.plot([0,1], [0,1], 'k--', label='Идеальная калибровка')
ax.set_xlabel('Предсказанная вероятность ДТП')
ax.set_ylabel('Реальная частота ДТП')
ax.set_title('Калибровочная кривая (частотная модель)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'calibration_curve_detailed.png', dpi=150)
plt.close()
print("4. Калибровочная кривая сохранена")

# -----------------------------------------------------------------------------
# 5. Распределение предсказанных вероятностей ДТП (для claim и non-claim)
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
train_claim = train[train['is_claim'] == 1]
train_noclaim = train[train['is_claim'] == 0]
ax.hist(train_noclaim['prob_claim'], bins=100, alpha=0.5, density=True, label='Нет ДТП', color='blue')
ax.hist(train_claim['prob_claim'], bins=100, alpha=0.5, density=True, label='Есть ДТП', color='red')
ax.axvline(x=CLAIM_RATE, color='black', linestyle='--', label=f'Средняя частота = {CLAIM_RATE:.3%}')
ax.set_xlabel('Предсказанная вероятность ДТП')
ax.set_ylabel('Плотность')
ax.set_title('Распределение вероятностей (train OOF)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'prob_distribution.png', dpi=150)
plt.close()
print("5. Распределение вероятностей сохранено")

# -----------------------------------------------------------------------------
# 6. Feature Importance (уже есть, но сделаем красивый горизонтальный барплот)
# -----------------------------------------------------------------------------
fi_top20 = fi.head(20).sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(fi_top20['feature'], fi_top20['importance'], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importance (Frequency Model)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'feature_importance_horizontal.png', dpi=150)
plt.close()
print("6. Feature Importance сохранён")

# -----------------------------------------------------------------------------
# 7. Lift-кривая (кумулятивная доля событий vs доля полисов)
# -----------------------------------------------------------------------------
# Сортируем полисы по убыванию prob_claim
train_sorted = train.sort_values('prob_claim', ascending=False).reset_index(drop=True)
train_sorted['cumulative_claims'] = train_sorted['is_claim'].cumsum()
train_sorted['cumulative_share_policies'] = (np.arange(len(train_sorted)) + 1) / len(train_sorted)
train_sorted['cumulative_share_claims'] = train_sorted['cumulative_claims'] / train_sorted['is_claim'].sum()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(train_sorted['cumulative_share_policies'], train_sorted['cumulative_share_claims'], 
        label='Модель', color='darkorange', linewidth=2)
ax.plot([0,1], [0,1], 'k--', label='Случайная модель')
ax.set_xlabel('Доля полисов')
ax.set_ylabel('Доля страховых случаев')
ax.set_title('Lift-кривая (накопление событий)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'lift_curve.png', dpi=150)
plt.close()
print("7. Lift-кривая сохранена")

# -----------------------------------------------------------------------------
# 8. Распределение expected_severity (тяжесть)
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(train['expected_severity'], bins=100, alpha=0.6, label='Ожидаемая тяжесть (train)', color='green')
ax.hist(train.loc[train['is_claim']==1, 'claim_amount'], bins=100, alpha=0.4, label='Реальная тяжесть (claim)', color='red')
ax.set_xlabel('Сумма (тг)')
ax.set_ylabel('Частота')
ax.set_title('Распределение тяжести ущерба')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'severity_distribution.png', dpi=150)
plt.close()
print("8. Распределение тяжести сохранено")

# -----------------------------------------------------------------------------
# 9. Региональные поправочные коэффициенты (severity)
# -----------------------------------------------------------------------------
if len(region_factor_map) > 0:
    region_factors = pd.DataFrame(list(region_factor_map.items()), columns=['region_id', 'factor'])
    region_factors = region_factors.sort_values('factor')
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(region_factors['region_id'].astype(str), region_factors['factor'], color='teal')
    ax.axhline(y=1.0, color='red', linestyle='--', label='Без поправки (1.0)')
    ax.set_xlabel('Region ID')
    ax.set_ylabel('Поправочный коэффициент')
    ax.set_title('Региональные поправки severity (train)')
    ax.legend()
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'figures' / 'region_factors.png', dpi=150)
    plt.close()
    print("9. Региональные поправки сохранены")
else:
    print("9. Нет региональных поправок")

# -----------------------------------------------------------------------------
# 10. Проверка ограничения ×3 (распределение отношения premium_new / premium)
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
ratio_test = test_pol['premium_new'] / test_pol['premium']
ax.hist(ratio_test, bins=100, alpha=0.7, color='purple')
ax.axvline(x=3.0, color='red', linestyle='--', label='Ограничение ×3')
ax.set_xlabel('Premium_new / Premium')
ax.set_ylabel('Частота')
ax.set_title('Распределение коэффициента изменения премии (тест)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'premium_ratio_test.png', dpi=150)
plt.close()
print("10. Ограничение ×3 проверено")

# -----------------------------------------------------------------------------
# 11. Сравнение распределения признаков в train и holdout (топ-5 важных)
# -----------------------------------------------------------------------------
if 'holdout_pol' in locals() and len(holdout_pol) > 0:
    top5_features = fi.head(5)['feature'].values
    fig, axes = plt.subplots(1, len(top5_features), figsize=(18, 5))
    for idx, feat in enumerate(top5_features):
        if feat in train.columns and feat in train_holdout.columns:
            axes[idx].hist(train[feat], bins=50, alpha=0.5, density=True, label='Train', color='blue')
            axes[idx].hist(train_holdout[feat], bins=50, alpha=0.5, density=True, label='Holdout', color='orange')
            axes[idx].set_title(feat)
            axes[idx].legend()
    plt.suptitle('Сравнение распределений топ-5 признаков (train vs holdout)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'figures' / 'feature_distribution_train_holdout.png', dpi=150)
    plt.close()
    print("11. Сравнение признаков train/holdout сохранено")
else:
    print("11. Holdout не найден, график пропущен")

# -----------------------------------------------------------------------------
# 12. ROC-кривая (по OOF предсказаниям)
# -----------------------------------------------------------------------------
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_freq, train['prob_claim'])
roc_auc = roc_auc_score(y_freq, train['prob_claim'])
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc:.4f})', color='darkorange', linewidth=2)
ax.plot([0,1], [0,1], 'k--', label='Случайная модель')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-кривая (частотная модель, OOF)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'roc_curve.png', dpi=150)
plt.close()
print("12. ROC-кривая сохранена")

# -----------------------------------------------------------------------------
# 13. График изменения LR в процессе калибровки scale (итерации)
# -----------------------------------------------------------------------------
# Если вы сохраняли историю итераций, можно её восстановить, но здесь просто покажем финальный scale
fig, ax = plt.subplots(figsize=(6, 6))
ax.text(0.5, 0.5, f'Финальный scale = {scale:.4f}\nLR = {lr_after:.2%}', 
        ha='center', va='center', fontsize=16, transform=ax.transAxes)
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.axis('off')
ax.set_title('Результат калибровки scale')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'scale_calibration_result.png', dpi=150)
plt.close()
print("13. Результат калибровки scale сохранён")

print("\n✅ Все графики сохранены в папку:", OUTPUT_DIR / "figures")


ГЕНЕРАЦИЯ ГРАФИКОВ
1. Распределение премий сохранено
2. LR по группам сохранён
3. Сравнение LR train/holdout сохранено
4. Калибровочная кривая сохранена
5. Распределение вероятностей сохранено
6. Feature Importance сохранён
7. Lift-кривая сохранена
8. Распределение тяжести сохранено
9. Региональные поправки сохранены
10. Ограничение ×3 проверено
11. Сравнение признаков train/holdout сохранено
12. ROC-кривая сохранена
13. Результат калибровки scale сохранён

✅ Все графики сохранены в папку: output_xgb_final\figures


In [7]:
# =============================================================================
# IV / WoE ANALYSIS — отдельная ячейка после основного pipeline
# Не меняет модель и не влияет на submission
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Если OUTPUT_DIR уже есть из основного кода — используем его
try:
    FIG_DIR = OUTPUT_DIR / "figures"
except NameError:
    FIG_DIR = Path("output_xgb_final") / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. Функция расчёта IV
# -----------------------------------------------------------------------------
def calculate_iv_for_feature(df, feature, target, bins=10, eps=1e-6):
    """
    IV для одного признака.
    target: 1 = claim, 0 = no claim
    good = no claim
    bad = claim
    """

    temp = df[[feature, target]].copy()
    temp[feature] = temp[feature].replace([np.inf, -np.inf], np.nan)

    # Если признак числовой и имеет много уникальных значений — бинируем
    if pd.api.types.is_numeric_dtype(temp[feature]) and temp[feature].nunique(dropna=True) > bins:
        try:
            temp["bin"] = pd.qcut(temp[feature], q=bins, duplicates="drop")
        except ValueError:
            temp["bin"] = temp[feature].fillna("Missing")
    else:
        temp["bin"] = temp[feature].astype("object").fillna("Missing")

    grouped = temp.groupby("bin", observed=False)[target].agg(["count", "sum"])
    grouped = grouped.rename(columns={"sum": "bad"})
    grouped["good"] = grouped["count"] - grouped["bad"]

    total_good = grouped["good"].sum()
    total_bad = grouped["bad"].sum()

    # защита от деления на ноль
    grouped["dist_good"] = grouped["good"] / max(total_good, eps)
    grouped["dist_bad"] = grouped["bad"] / max(total_bad, eps)

    grouped["woe"] = np.log((grouped["dist_good"] + eps) / (grouped["dist_bad"] + eps))
    grouped["iv_component"] = (grouped["dist_good"] - grouped["dist_bad"]) * grouped["woe"]

    iv = grouped["iv_component"].sum()

    return iv


# -----------------------------------------------------------------------------
# 2. Считаем IV по признакам модели
# -----------------------------------------------------------------------------

target_col = "is_claim"

# Берём признаки из модели
# feature_cols уже создан в основном pipeline
iv_features = []

for col in feature_cols:
    if col in train.columns:
        iv_features.append(col)

iv_results = []

for col in iv_features:
    try:
        iv_value = calculate_iv_for_feature(train, col, target_col, bins=10)
        iv_results.append({
            "feature": col,
            "IV": iv_value
        })
    except Exception as e:
        print(f"Пропущен признак {col}: {e}")

iv_df = pd.DataFrame(iv_results)
iv_df = iv_df.sort_values("IV", ascending=False).reset_index(drop=True)

# Сохраняем таблицу
iv_df.to_csv(FIG_DIR / "iv_results.csv", index=False)

print("Top IV features:")
print(iv_df.head(20).to_string(index=False))


# -----------------------------------------------------------------------------
# 3. График топ-20 IV
# -----------------------------------------------------------------------------

top_n = 20
plot_df = iv_df.head(top_n).sort_values("IV", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(plot_df["feature"], plot_df["IV"])

plt.axvline(0.02, linestyle="--", linewidth=1, label="IV = 0.02")
plt.axvline(0.10, linestyle="--", linewidth=1, label="IV = 0.10")
plt.axvline(0.30, linestyle="--", linewidth=1, label="IV = 0.30")

plt.title("Information Value (IV): Top 20 признаков")
plt.xlabel("IV")
plt.ylabel("Признак")
plt.legend()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.savefig(FIG_DIR / "iv_top20.png", dpi=150)
plt.show()

print(f"\nГрафик сохранён: {FIG_DIR / 'iv_top20.png'}")
print(f"Таблица сохранена: {FIG_DIR / 'iv_results.csv'}")

Top IV features:
          feature       IV
       pol_avg_bm 0.150559
  pol_max_bm_risk 0.139085
        region_id 0.100968
          bm_risk 0.073090
bonus_malus_clean 0.070696
      pol_min_exp 0.059356
       bm_is_good 0.051716
      region_x_bm 0.043063
  vehicle_x_power 0.037852
  experience_year 0.031906
  vehicle_type_id 0.031161
       bm_is_base 0.030972
        exp_group 0.026293
      power_x_new 0.023361
    power_per_vol 0.022220
      SCORE_2_avg 0.019381
    is_new_driver 0.019151
   pol_driver_cnt 0.016712
      SCORE_8_avg 0.016582
    drivers_x_new 0.015716

График сохранён: output_xgb_final\figures\iv_top20.png
Таблица сохранена: output_xgb_final\figures\iv_results.csv
